In [ ]:
import os, sys, subprocess
if not os.path.exists("EAGLE"):
    subprocess.run(["git", "clone", "https://github.com/SafeAILab/EAGLE.git"], check=True)
sys.path.insert(0, os.path.abspath("EAGLE"))

In [ ]:
import time, json, random, math
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Sequence, Optional, List

import torch.nn.functional as F
from tqdm.auto import tqdm

from eagle.model.ea_model import EaModel
from eagle.model import utils as eagle_utils

In [ ]:
@dataclass
class DDDConfig:
    max_depth: int = 11
    top_k: int = 10
    total_tokens: int = 60
    check_steps: Sequence[int] = field(default_factory=lambda: (5, 7, 9))
    continue_threshold: float = -3.0

def install_ddd(model, cfg: DDDConfig) -> None:
    draft = getattr(model, "ea_layer", None) or getattr(model, "draft", None)
    draft._ddd_cfg = cfg
    draft.topK_genrate = _ddd_topK_genrate.__get__(draft, type(draft))


def uninstall_ddd(model) -> None:
    draft = getattr(model, "ea_layer", None) or getattr(model, "draft", None)
    if draft is not None and hasattr(draft, "_orig_topK_genrate"):
        draft.topK_genrate = draft._orig_topK_genrate
        del draft._orig_topK_genrate

In [ ]:
import inspect, textwrap, re

@dataclass
class DDDConfig:
    max_depth: int = 11
    check_steps: tuple = (5, 7, 9)
    continue_threshold: float = -3.0
    enable_early_stop: bool = False
    verbose: bool = False


def install_ddd(model, cfg: DDDConfig) -> None:
    """Patch the draft to enable DDD. Safe mode: just bump self.depth."""
    draft = getattr(model, "ea_layer", None) or getattr(model, "draft", None)
    if draft is None:
        raise RuntimeError("Could not locate EAGLE draft network.")
    if hasattr(draft, "_orig_depth") or hasattr(draft, "_orig_topK_genrate"):
        return

    draft._orig_depth = draft.depth
    draft.depth = cfg.max_depth
    if cfg.verbose:
        print(f"[DDD] safe mode: depth {draft._orig_depth} -> {draft.depth}")

    if not cfg.enable_early_stop:
        return

    src = inspect.getsource(draft.__class__.topK_genrate)
    src = textwrap.dedent(src)

    for_re = re.compile(r"^(?P<indent>[ \t]*)for i in range\(depth\):\s*$", re.MULTILINE)
    m = for_re.search(src)

    indent = m.group("indent")
    body = indent + "    "
    new_loop = (
        f"{indent}for i in range({cfg.max_depth}):\n"
        f"{body}# --- DDD early stop ---\n"
        f"{body}if (i + 1) in {tuple(cfg.check_steps)} and "
        f"torch.logsumexp(scores, dim=0).item() < {cfg.continue_threshold}:\n"
        f"{body}    break"
    )
    src = for_re.sub(new_loop, src, count=1)
    src = re.sub(r"^[ \t]*@torch\.no_grad\(\)\s*\n", "", src, count=1)
    src = src.replace("def topK_genrate(", "def _ddd_topK_genrate_patched(", 1)

    if cfg.verbose:
        print("[DDD] patched topK_genrate source (first 60 lines):")
        for line_no, line in enumerate(src.splitlines()[:60], 1):
            print(f"  {line_no:3d}: {line}")
        print("  ...")

    orig_module = inspect.getmodule(draft.__class__)
    namespace = dict(orig_module.__dict__)
    namespace["torch"] = torch
    try:
        exec(compile(src, "<ddd_patched>", "exec"), namespace)
    except Exception as e:
        print(f"[DDD] WARNING: patched source failed to compile ({e}); "
              "falling back to safe mode (depth-only).")
        return
    fn = namespace["_ddd_topK_genrate_patched"]
    fn = torch.no_grad()(fn)

    draft._orig_topK_genrate = draft.topK_genrate
    draft.topK_genrate = fn.__get__(draft, type(draft))
    if cfg.verbose:
        print(f"[DDD] full mode: early-stop installed, check_steps={cfg.check_steps}, "
              f"threshold={cfg.continue_threshold}")


def uninstall_ddd(model) -> None:
    draft = getattr(model, "ea_layer", None) or getattr(model, "draft", None)
    if draft is None:
        return
    if hasattr(draft, "_orig_topK_genrate"):
        draft.topK_genrate = draft._orig_topK_genrate
        del draft._orig_topK_genrate
    if hasattr(draft, "_orig_depth"):
        draft.depth = draft._orig_depth
        del draft._orig_depth

In [ ]:
@dataclass
class CactusConfig:
    delta: float = 1.0
    verbose = False


def _make_cactus_evaluator(cfg: CactusConfig, fallback):
    def evaluator(logits, candidates, logits_processor, *args, **kwargs):
        if logits_processor is not None:
            return fallback(logits, candidates, logits_processor, *args, **kwargs)

        if logits.dim() == 4 and logits.shape[0] == 1:
            logits = logits.squeeze(0)
        if candidates.dim() == 3 and candidates.shape[0] == 1:
            candidates = candidates.squeeze(0)
        P, D, V = logits.shape

        target_logp = torch.log_softmax(logits, dim=-1)
        max_logp, _ = target_logp.max(dim=-1)
        draft_logp = target_logp.gather(-1, candidates.unsqueeze(-1)).squeeze(-1)
        node_kl = (max_logp - draft_logp).clamp(min=0.0)
        cum_kl = node_kl.cumsum(dim=-1)

        accept_mask = cum_kl < cfg.delta
        path_len = accept_mask.long().sum(dim=-1)

        max_len_t = path_len.max()
        max_len = int(max_len_t.item())
        if max_len == 0:
            best_p = 0
            sample_p = logits[best_p, 0].softmax(dim=-1)
            return best_p, 0, sample_p

        tail_kl = cum_kl[:, max_len - 1]
        tail_kl = tail_kl.masked_fill(path_len != max_len_t, float("inf"))
        best_p = int(tail_kl.argmin().item())

        end_d = min(max_len, D - 1)
        sample_p = logits[best_p, end_d].softmax(dim=-1)
        return best_p, max_len, sample_p
    return evaluator

In [ ]:
class RelaxedEaModel(EaModel):
    @torch.no_grad()
    def eagenerate_relaxed(
        self, input_ids, max_new_tokens=20, method="baseline",
        cactus_cfg=None, ddd_cfg=None, temperature=0.0, **kwargs,
    ):
        if method not in ("baseline", "cactus", "ddd", "cactus_ddd"):
            raise ValueError(f"unknown method {method!r}")

        use_ddd = method in ("ddd", "cactus_ddd")
        use_cactus = method in ("cactus", "cactus_ddd")
        if use_cactus and cactus_cfg is None:
            cactus_cfg = CactusConfig()
        if use_ddd and ddd_cfg is None:
            ddd_cfg = DDDConfig()

        with self._patches(use_ddd, ddd_cfg, use_cactus, cactus_cfg):
            output_ids = self.eagenerate(
                input_ids, temperature=temperature,
                max_new_tokens=max_new_tokens, **kwargs,
            )
        return output_ids

    @contextmanager
    def _patches(self, use_ddd, ddd_cfg, use_cactus, cactus_cfg):
        if use_ddd:
            install_ddd(self, ddd_cfg)

        patched_modules = []
        if use_cactus:
            import importlib, sys
            new_eval = _make_cactus_evaluator(cactus_cfg, fallback=eagle_utils.evaluate_posterior)
            for mod_name, mod in list(sys.modules.items()):
                if mod is None or not mod_name.startswith("eagle"):
                    continue
                if getattr(mod, "evaluate_posterior", None) is not None:
                    patched_modules.append((mod, mod.evaluate_posterior))
                    mod.evaluate_posterior = new_eval

        try:
            yield
        finally:
            if use_ddd:
                uninstall_ddd(self)
            for mod, orig in patched_modules:
                mod.evaluate_posterior = orig

In [ ]:
from datasets import load_dataset

@dataclass
class CodeSample:
    raw_text: str
    input_ids: torch.Tensor


CODE_DATASET = "bigcode/the-stack-smol"
CODE_LANGUAGES = ("python", "javascript", "go", "rust", "c++")


def load_stack_v2_samples(tokenizer, n_samples=15, max_tokens=2048, min_tokens=256,
                          seed=0, languages=CODE_LANGUAGES):
    per_lang = max(4, (n_samples * 2) // len(languages))

    pooled = []
    for lang in languages:
        ds = load_dataset(CODE_DATASET, data_dir=f"data/{lang}", split="train", streaming=True)
        ds = ds.shuffle(seed=seed, buffer_size=1000)
        for item in ds.take(per_lang):
            text = item.get("content") or ""
            if text:
                pooled.append((lang, text))

    rng = random.Random(seed)
    rng.shuffle(pooled)

    samples = []
    skipped = 0
    for lang, text in pooled:
        if len(samples) == n_samples:
            break
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids
        if ids.shape[1] < min_tokens:
            skipped += 1
            continue
        samples.append(CodeSample(raw_text=text, input_ids=ids[:, :max_tokens]))

    return samples

In [ ]:
def token_levenshtein(a, b):
    a, b = list(a), list(b)
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m + 1))
    curr = [0] * (m + 1)
    for i in range(1, n + 1):
        curr[0] = i; ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            curr[j] = min(curr[j - 1] + 1, prev[j] + 1, prev[j - 1] + cost)
        prev, curr = curr, prev
    return prev[m]


def generate_and_time(model, prompt_ids, gen_fn, gen_tokens):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    out_ids = gen_fn(prompt_ids, gen_tokens)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dur = time.perf_counter() - start

    out_ids = out_ids[0].tolist() if hasattr(out_ids, "tolist") else list(out_ids)
    prompt_len = prompt_ids.shape[1]
    return out_ids[prompt_len:prompt_len + gen_tokens], dur

In [ ]:
MODEL_CONFIGS = {
    "7B": {
        "base_model_path": "lmsys/vicuna-7b-v1.3",
        "ea_model_path":   "yuhuili/EAGLE-Vicuna-7B-v1.3",
        "total_token": -1,
    },
    "13B": {
        "base_model_path": "lmsys/vicuna-13b-v1.3",
        "ea_model_path":   "yuhuili/EAGLE-Vicuna-13B-v1.3",
        "total_token": -1,
    },
    "33B": {
        "base_model_path": "lmsys/vicuna-33b-v1.3",
        "ea_model_path":   "yuhuili/EAGLE-Vicuna-33B-v1.3",
        "total_token": -1,
    },
}

DDD_DEFAULTS = DDDConfig(max_depth=11, check_steps=(5, 7, 9), continue_threshold=-3.0)

METHOD_VARIANTS = [
    ("baseline",    None,  "Baseline"),
    ("cactus",      0.1,   "Cactus 0.1"),
    ("cactus",      1.0,   "Cactus 1.0"),
    ("ddd",         None,  "DDD"),
    ("cactus_ddd",  0.1,   "Cactus 0.1 + DDD"),
    ("cactus_ddd",  1.0,   "Cactus 1.0 + DDD"),
]

N_SAMPLES = 1
POSITIONS_PER_SAMPLE = 32
GEN_TOKENS = 20
MAX_TOKENS = 256
SEED = 0
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
def run_one_cell(model, samples, method, kl_threshold, label, size, *,
                 positions_per_sample=POSITIONS_PER_SAMPLE, gen_tokens=GEN_TOKENS, seed=SEED):
    cactus_cfg = CactusConfig(delta=kl_threshold) if "cactus" in method else None
    ddd_cfg    = DDD_DEFAULTS                     if "ddd"    in method else None

    rng = random.Random(seed)
    accs, durs = [], []
    n_tokens_total = 0

    for sample in tqdm(samples, desc=f"{size}/{label}", leave=False):
        full = sample.input_ids
        L = full.shape[1]
        max_cut = L - gen_tokens
        if max_cut < 1:
            continue

        if positions_per_sample == -1:
            cuts = list(range(1, max_cut))
        else:
            cuts = sorted(rng.sample(range(1, max_cut), min(positions_per_sample, max_cut - 1)))

        for cut in cuts:
            prompt_ids = full[:, :cut].to(model.base_model.device)
            ground_truth = full[0, cut:cut + gen_tokens].tolist()

            def gen_fn(p, n):
                return model.eagenerate_relaxed(
                    p, max_new_tokens=n, method=method,
                    cactus_cfg=cactus_cfg, ddd_cfg=ddd_cfg, temperature=0.0,
                )

            new_ids, dur = generate_and_time(model, prompt_ids, gen_fn, gen_tokens)
            d = token_levenshtein(new_ids, ground_truth)
            acc = 100.0 * max(0.0, min(1.0, 1.0 - d / gen_tokens))
            accs.append(acc); durs.append(dur); n_tokens_total += len(new_ids)

    return {
        "model_size": size,
        "method": method,
        "kl_threshold": kl_threshold,
        "label": label,
        "n_generations": len(accs),
        "n_tokens_total": n_tokens_total,
        "token_accuracy_pct": sum(accs) / len(accs) if accs else 0.0,
        "ms_per_token": 1000.0 * sum(durs) / n_tokens_total if n_tokens_total else 0.0,
    }

In [ ]:
from huggingface_hub import login
hf_token = "" # Insert HF Token
login(token=hf_token)

In [ ]:
MODEL_SIZES = ["7B", "13B", "33B"]
all_results = []

for size in MODEL_SIZES:
    cfg = MODEL_CONFIGS[size]
    print(f"\n=== Loading Vicuna {size} (EAGLE-2) ===")
    model = RelaxedEaModel.from_pretrained(
        base_model_path=cfg["base_model_path"],
        ea_model_path=cfg["ea_model_path"],
        use_eagle3=False,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
        total_token=cfg["total_token"],
    )
    model.eval()

    samples = load_stack_v2_samples(model.tokenizer, n_samples=N_SAMPLES,
                                     max_tokens=MAX_TOKENS, seed=SEED)
    print(f"loaded {len(samples)} Stack V2 samples")

    for method, kl, label in METHOD_VARIANTS:
        result = run_one_cell(model, samples, method, kl, label, size)
        all_results.append(result)
        kl_tag = f"_{kl}" if kl is not None else ""
        out_path = RESULTS_DIR / f"vicuna_{size.lower()}_{method}{kl_tag}.json"
        out_path.write_text(json.dumps(result, indent=2))
        print(f"  {label:20s} acc={result['token_accuracy_pct']:5.1f}%  "
              f"ms/tok={result['ms_per_token']:6.2f}  -> {out_path}")

    del model
    torch.cuda.empty_cache()

print(f"\nsweep done: {len(all_results)} cells")